# Vision Transformr

**0 前言**<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.1 为什么图像任务也可以使用Transformer？<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.2 从CNN到ViT：视觉建模思路的变化<br>

**1 ViT模型的核心思想与整体结构**<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.1 ViT到底想解决什么问题？<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.2 ViT整体结构与数据流<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.3 ViT和原始Transformer的关系<br>

**2 ViT模型的输入：从图像到Patch序列**<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.1 图像为什么要切成Patch？<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.2 Patch Flatten与Linear Projection<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.3 图像输入在ViT中的张量形状变化<br>

<font color="red">**3 分类标记CLS Token与分类输出**</font><br>
&nbsp;&nbsp;&nbsp;&nbsp;3.1 为什么要额外加入一个分类标记？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.2 CLS Token如何汇总整张图像的信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.3 分类头MLP Head如何输出类别概率？<br>

**4 ViT中的位置编码**<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.1 图像Patch为什么也需要位置信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.2 ViT中的可学习位置编码<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.3 位置编码与图像二维结构的关系<br>

**5 ViT中的Transformer Encoder主干**<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.1 ViT中的多头自注意力机制<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.2 ViT中的前馈神经网络MLP Block<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.3 残差连接与Layer Normalization<br>

**6 ViT的训练、特点与局限**<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.1 ViT为什么通常需要较大数据量？<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.2 ViT和CNN的归纳偏置差异<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.3 ViT的优势、局限与常见变体<br>


## 3 分类标记CLS Token与分类输出

前一章我们已经看到，一张图像进入ViT之后，会先被切成若干个patch，再经过flatten和线性投影，最终变成一串patch embedding。到这一步为止，模型已经拥有了一整条输入序列。

不过，这时还有一个非常关键的问题没有解决：

**如果这是一项图像分类任务，模型最后应该拿这串序列里的哪一部分去代表“整张图”的信息？**

这个问题看起来简单，实际上非常关键。因为Transformer内部会输出一串更新后的token表示，也就是说，每个patch最后都会得到一个新的向量。但图像分类需要的不是“每个patch分别属于什么类别”，而是“整张图属于什么类别”。所以模型必须想办法把整张图的信息汇总成一个可用于分类的全局表示。

ViT处理这个问题的方式并不是在最后简单地把所有patch向量平均一下，而是采用了一个非常有代表性的设计：**在输入序列最前面额外加入一个特殊的标记，也就是CLS Token。**

这一章我们就专门来讲这个特殊token到底是干什么的，它为什么要存在，以及它最后是怎样参与分类输出的。


### 3.1 为什么要额外加入一个分类标记？

我们先把问题说得更具体一点。

在ViT中，一张图像会先被切成很多patch，然后这些patch会变成一串token序列，例如：

$$
[patch_1, patch_2, patch_3, \ldots, patch_N]
$$

经过Transformer Encoder之后，模型仍然会输出一串同样长度的表示：

$$
[h_1, h_2, h_3, \ldots, h_N]
$$

这里的每个 $h_i$ 都对应一个patch经过多层计算之后的最终表示。

问题在于：**图像分类需要一个“整图级别”的输出，但Encoder天然给出的是“一串patch级别”的输出。那到底该拿哪一个向量去做最后分类？**

如果这个问题不先解决，后面的分类头就无从谈起。

一种最直接的想法是：既然有很多patch向量，那就把它们做平均，或者求和，最后得到一个整体向量。这种思路并不是完全不行，后来的不少模型里也确实会使用全局平均池化这类方法。

但是ViT采用的是另一种更有结构感的办法：**专门在序列里安排一个“负责汇总全局信息的位置”，让模型在计算过程中主动把整张图的重要信息聚合到这个位置上。** 这个特殊位置对应的输入向量，就是`CLS Token`。

也就是说，ViT不是等所有patch都算完以后，再临时想办法把它们压成一个向量；而是从一开始就在输入序列里放进一个“全局代表”，让它和所有patch一起参与后续的注意力计算。

这时序列就不再只是：

$$
[patch_1, patch_2, patch_3, \ldots, patch_N]
$$

而会变成：

$$
[CLS, patch_1, patch_2, patch_3, \ldots, patch_N]
$$

这里最前面的`CLS`，就是额外插入的分类标记。

那么，为什么这种设计是合理的？

关键在于Transformer里的自注意力机制。由于`CLS Token`和其他patch token一起进入Encoder，在每一层注意力计算中，它都可以和整条序列中的其他token发生交互。换句话说，`CLS Token`并不是一个静止不动的占位符，而是会在多层计算中不断吸收来自各个patch的信息，逐渐形成一个更偏向“整张图摘要”的表示。

从这个角度看，`CLS Token`更像是模型专门预留出来的一个“信息汇总站”。它本身一开始并不是某个具体patch，也不对应图像中的某个真实局部区域；它的作用是作为一个全局性的容器，去收集整张图的重要信息。

这里尤其要注意，**CLS Token不是从原始图像里切出来的patch。** 这是初学者最容易误解的点之一。更准确地说：

- 普通patch token来自输入图像的真实局部区域
- CLS Token不是图像中的某个区域
- 它是人为额外加入到序列前面的一个可学习向量

也就是说，CLS Token属于模型设计的一部分，而不是输入图像本身的一部分。

这时你可能会问：**既然CLS Token一开始不是来自图像内容，那它怎么可能代表整张图？**

答案就在“训练”和“注意力交互”这两个词里。虽然CLS Token一开始只是一个可学习向量，但在训练过程中，它会和所有patch token一起被不断更新；而在每一层自注意力中，它又有机会从其他patch那里收集信息。久而久之，模型就会学会把“与分类最相关的全局信息”逐步压到这个特殊位置上。

你可以把它想象成班级里专门负责做总结的同学。其他同学分别掌握了不同片段的信息，而这个“总结者”会不断听取别人发言，最后形成一个更全面的结论。CLS Token在ViT里的角色，就有点类似这种“全局汇总位”。

从工程实现上看，引入CLS Token还有一个直接好处：**模型最后做分类时，就可以统一地取序列最前面那个位置的输出向量，而不必临时再设计额外的汇聚规则。**

这使得整个分类流程更整齐：

1. 在输入序列前面加一个CLS Token。
2. 让它和所有patch一起经过Transformer Encoder。
3. 取最终的CLS输出向量送入分类头。

这样一来，“谁来代表整张图”这个问题，就被转换成了一个非常清晰的结构设计问题，而不是留到最后再临时处理。

到这里，我们已经回答了“为什么要额外加入一个分类标记”这个问题：

- 因为图像分类需要一个整图级别的全局表示
- 而Transformer Encoder天然输出的是一串patch级别表示
- 所以需要人为加入一个专门负责汇总全局信息的位置
- 这个位置就是CLS Token

不过，知道“为什么要有它”还不够。下一节我们还要更进一步看：**CLS Token到底是怎样在注意力计算中逐步汇总整张图信息的？** 也就是说，我们要从“设计动机”进一步走向“具体工作机制”。


### 3.2 CLS Token如何汇总整张图像的信息？

上一节我们已经知道，ViT之所以要加入`CLS Token`，是因为图像分类需要一个整图级别的全局表示，而Transformer Encoder天然输出的却是一串patch级别的表示。

但这时还有一个真正关键的问题：

**CLS Token明明不是从图像里切出来的patch，它到底是怎样一步步获得整张图的信息的？**

要回答这个问题，就必须把它放回Transformer Encoder的计算过程里看。核心答案其实可以先概括成一句话：

**CLS Token之所以能够汇总整张图的信息，是因为它会和所有patch token一起参与自注意力计算，并在一层又一层的交互中不断吸收来自各个patch的信息。**

这句话听起来有点抽象，我们把它拆开来看。

#### 3.2.1 先从输入序列的形式看起

加入CLS Token之后，一张图像对应的输入序列可以写成：

$$
[x_{cls}, x_1, x_2, x_3, \ldots, x_N]
$$

其中：

- $x_{cls}$ 表示CLS Token对应的输入向量
- $x_1, x_2, \ldots, x_N$ 表示 $N$ 个patch token的输入向量

这里尤其要注意，**CLS Token虽然放在序列最前面，但它和其他patch token在Encoder里是平等参与计算的。** 它不是在最后单独拿出来处理，也不是只在最后一步才与其他token接触，而是一开始就进入了整条序列。

也正因为如此，它才能在每一层里持续和其他patch交互。

#### 3.2.2 在自注意力里，CLS Token可以读取所有patch的信息

在一层自注意力中，序列中的每个token都会根据当前自己的表示生成查询向量（Query）、键向量（Key）和值向量（Value），然后去和整条序列中的其他token计算相关性，决定自己应该从哪些位置接收更多信息。

这句话放到CLS Token身上，含义就是：

- CLS Token也会生成自己的Query
- 它会拿这个Query去和所有token的Key计算相似度
- 这些token既包括各个patch，也包括它自己
- 然后它会按这些注意力权重，对所有token的Value做加权求和

于是，CLS Token在这一层更新后的表示，就不再只是它原来的那个初始向量，而是一个**融合了整条序列信息的新表示**。

如果用符号粗略地写，它的更新过程可以理解成：

$$
h_{cls}^{(1)} = \sum_{j=cls}^{N} \alpha_{cls,j} v_j
$$

这里不需要你现在就死记这个公式，更重要的是理解它表达的意思：

- $h_{cls}^{(1)}$ 表示第一层之后CLS Token的新表示
- $v_j$ 表示序列中第 $j$ 个token对应的Value向量
- $\alpha_{cls,j}$ 表示CLS Token对第 $j$ 个token分配了多少注意力权重

换句话说，**CLS Token的新表示，本质上是从整条序列中按重要性加权汇总出来的结果。**

这正是它能够承担“整图摘要”角色的根本原因。

#### 3.2.3 一个更直观的理解：CLS Token像一个不断做总结的全局位置

如果只看公式，很多初学者会觉得抽象。我们换一个更直观的说法。

可以把整条patch序列想象成一组分别掌握局部信息的观察者：

- 有的patch更像在描述边缘
- 有的patch更像在描述纹理
- 有的patch落在目标物体主体上，信息更关键
- 有的patch落在背景上，信息可能较弱

而CLS Token更像是一个专门负责做“全局总结”的位置。每经过一层注意力，它都会去参考其他patch当前提供的信息，并把其中对分类更重要的部分吸收到自己这里。于是层数越深，它的表示就越不像一个孤立的初始向量，而越像一个融合了全图上下文的摘要向量。

这里可以特别注意一个细节：**CLS Token不是只和某一个patch交互，而是理论上可以和所有patch建立联系。** 这意味着它汇总的是全局信息，而不只是某个局部片段的信息。

#### 3.2.4 为什么说它是逐层汇总，而不是一次性拿到全图信息？

另一个容易误解的点是，很多人会以为CLS Token只要在第一层和所有patch做一次注意力，就已经立刻拿到了整张图的全部信息。这个理解不够准确。

更准确地说，**CLS Token对整图信息的汇总是逐层进行的。**

原因在于，patch token自己也在每一层中不断更新。第一层时，每个patch的表示可能还更接近原始局部信息；经过后续层之后，这些patch表示本身已经融合了更多来自其他区域的上下文。于是CLS Token在更深层再去读取这些patch时，它拿到的就不只是“原始局部块”，而是“已经带有上下文的局部表示”。

所以，CLS Token的全局摘要能力不是一蹴而就的，而是在“patch彼此交互”和“CLS读取patch信息”这两个过程反复交织之后逐渐形成的。

如果把这个过程说得更完整一点，就是：

1. 每个patch token在层内会和其他patch交互，更新自己的表示。
2. CLS Token也会读取这些token的信息，形成自己的新表示。
3. 到了下一层，所有token都带着上层更新后的表示继续交互。
4. 因此，层数越深，CLS中聚合的信息通常越接近面向任务的全局语义。

#### 3.2.5 这里的汇总不是简单平均，而是有选择地聚合

还有一个非常值得强调的点：**CLS Token汇总信息的过程，不是把所有patch一视同仁地平均起来。**

如果只是简单平均，那么所有patch对最终结果的贡献会被机械地设成一样大。但在真实图像里，不同区域的重要性显然不一样。例如，在“猫”的分类任务里，猫脸、耳朵、身体轮廓对应的patch，通常会比纯背景patch更重要。

自注意力的意义就在这里：CLS Token可以通过注意力权重，自适应地决定“更该从哪些patch吸收信息”。也就是说，它做的是一种**有选择的、数据驱动的加权汇总**，而不是固定规则下的平均压缩。

#### 3.2.6 一个小例子帮助建立直觉

假设一张图像被切成4个patch：

$$
[x_{cls}, x_1, x_2, x_3, x_4]
$$

其中：

- $x_1$ 和 $x_2$ 落在目标物体主体上
- $x_3$ 只包含少量边缘信息
- $x_4$ 主要是背景

那么在某一层注意力中，CLS Token就可能对这4个patch分配不同的注意力权重。例如，它可能更关注 $x_1$ 和 $x_2$，较少关注 $x_4$。这样一来，更新后的CLS表示就会更多吸收与分类相关的主体信息，而不是被大量无关背景平均稀释掉。

当然，这里的注意力权重不是人工指定的，而是在训练中自动学出来的。模型会根据任务目标，逐步学会哪些区域更值得让CLS去关注。

#### 3.2.7 这一节真正应该记住什么

到这里，我们可以把CLS Token的工作机制压缩成下面几句话：

- CLS Token从一开始就和所有patch token一起进入Transformer Encoder
- 在每一层自注意力中，它都可以和整条序列中的其他token交互
- 它会按注意力权重，从各个patch那里有选择地吸收信息
- 这种汇总不是一次性完成的，而是随着层数加深逐步形成的
- 最终，CLS Token会变成一个更适合代表整张图的全局摘要向量

理解了这一点之后，下一步就很自然了：既然最后CLS Token已经变成了一个整图表示，那么模型到底是怎样利用这个表示输出类别概率的？这就是下一节要讲的分类头MLP Head。


### 3.3 分类头MLP Head如何输出类别概率？

到上一节为止，我们已经知道了两件关键的事：

- ViT会在输入序列最前面加入一个CLS Token
- 经过多层Transformer Encoder之后，这个CLS Token会逐步汇总整张图的全局信息

那么接下来还差最后一步：

**既然CLS Token已经变成了整图表示，模型到底怎样把这个表示变成最终的分类结果？**

这一步就是分类头，也常写作`MLP Head`。

#### 3.3.1 先从最直接的问题看起：分类任务最终需要什么输出？

图像分类任务的目标，不是让模型输出一串token，也不是让模型重建图像，而是让模型回答一个非常明确的问题：

**这张图属于哪一个类别？**

例如，在一个猫狗分类任务里，最终输出可能只有两个类别：

- 猫
- 狗

而在ImageNet这类任务里，类别数可能有1000个。

这意味着，模型最后必须把整张图的表示映射成各个类别对应的分数。这个把高维表示变成类别分数的模块，就是分类头。

#### 3.3.2 分类头的输入是什么？

在ViT里，分类头通常接收的不是全部patch token，而是**最后一层Encoder输出中的CLS Token表示**。

如果把最后一层CLS Token的输出记为：

$$
h_{cls} \in \mathbb{R}^{d_{model}}
$$

那么它就是一个长度为 $d_{model}$ 的向量。

这里尤其要注意，**这个向量已经不再是初始的CLS Token，而是经过多层注意力和前馈网络更新后的最终全局表示。** 也正因为如此，它才有资格被拿去做整图分类。

#### 3.3.3 最简单的分类头：一层线性映射

很多情况下，分类头可以非常简单，甚至就是一个线性层。如果类别数记为 $K$，那么分类头可以把CLS向量映射成一个长度为 $K$ 的输出向量：

$$
z = h_{cls} W + b
$$

其中：

- $h_{cls}$ 是CLS Token的最终表示
- $W$ 是分类层的可训练权重矩阵
- $b$ 是偏置项
- $z$ 是输出的类别分数向量

如果更明确地看形状：

- $h_{cls}$ 的形状是 $d_{model}$
- $W$ 的形状是 $d_{model} \times K$
- $b$ 的形状是 $K$
- 输出 $z$ 的形状是 $K$

也就是说，分类头做的事情并不神秘，本质上就是：**把一个全局特征向量映射成每个类别对应的分数。**

这里的 $z$ 中通常会有 $K$ 个数，每个数对应一个类别。例如：

$$
z = [2.1, 0.3, -1.2]
$$

如果这是一个三分类任务，那么这三个数就分别对应三个类别的原始分数。

#### 3.3.4 这些输出分数就是概率吗？

这正是初学者最容易混淆的地方之一。答案是：**不是。**

分类头线性层直接输出的 $z$，通常叫做`logits`，也就是类别得分或未归一化分数。这些数可以是正的、负的，也不需要加起来等于1，因此它们本身还不能直接当作概率解释。

如果希望把这些分数变成真正的类别概率，通常还要再经过一个`Softmax`：

$$
p_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

这里：

- $z_i$ 表示第 $i$ 个类别的logit
- $p_i$ 表示第 $i$ 个类别对应的预测概率
- $K$ 表示类别总数

经过Softmax之后，所有类别的概率会满足：

- 每个 $p_i$ 都在 0 到 1 之间
- 所有类别概率加起来等于 1

这样我们才可以把输出解释为模型认为这张图属于各类别的概率分布。

#### 3.3.5 用一个小例子把logits和概率区分开

假设某张图像经过ViT和分类头之后，输出的logits是：

$$
z = [2.1, 0.3, -1.2]
$$

这三个数可能分别对应：

- 类别1：猫
- 类别2：狗
- 类别3：鸟

从这个结果里，我们可以先看出模型更偏向猫，因为 `2.1` 最大。但这三个数本身还不是概率。

经过Softmax之后，它可能变成类似：

$$
p = [0.82, 0.14, 0.04]
$$

这时我们才可以说：

- 模型认为这张图是猫的概率约为0.82
- 是狗的概率约为0.14
- 是鸟的概率约为0.04

所以，这里最好牢牢记住：

- `logits` 是分类头直接输出的原始分数
- `softmax之后的结果` 才是概率分布

#### 3.3.6 为什么有时叫MLP Head，而不只是Linear Head？

你可能会注意到，很多资料里会把分类头写成`MLP Head`，而不只是一个线性层。这是因为在具体实现中，分类头有时确实不止一层，有的模型会在最终输出前加上额外的线性层、激活函数，甚至加入LayerNorm等结构。

不过从入门理解上看，最核心的事情并没有变：

**分类头的本质任务，就是把CLS Token的最终表示映射成类别分数。**

所以你可以先把它理解成从全局表示到类别空间的最后一段映射。至于具体是单层线性头，还是更复杂一点的MLP结构，那属于实现细节上的差异。

#### 3.3.7 训练时模型是怎样学会正确分类的？

再往前走一步，训练时模型并不是只输出一个概率就结束了。它还会把这个预测结果和真实标签进行比较，并通过损失函数来调整参数。

在分类任务中，最常见的做法是使用交叉熵损失。它会鼓励模型：

- 给真实类别分配更高的分数和概率
- 给错误类别分配更低的分数和概率

而这个训练信号会反向传播回去，不仅更新分类头的参数，也会继续更新CLS Token相关表示、Encoder中的注意力层和前馈网络参数。也就是说，**分类头虽然位于最末端，但它的监督信号会反过来影响整个ViT。**

这也是为什么CLS Token会越来越擅长汇总与分类相关的信息，因为整个模型会在训练中一起协同优化。

#### 3.3.8 这一节真正应该记住什么

到这里，ViT中的分类输出流程就完整了。你应该抓住下面这条主线：

1. CLS Token在Encoder中逐层汇总整张图的信息。
2. 最后一层的CLS表示被当作整图的全局表示。
3. 分类头把这个全局表示映射成各类别的logits。
4. Softmax再把logits变成类别概率。

用一句话压缩，就是：

**ViT不是直接拿所有patch去做分类，而是先让CLS Token成为整图摘要，再由分类头根据这个摘要输出类别概率。**

到这里，CLS Token与分类输出这部分就完整了。接下来，一个自然的问题就是：既然输入序列里有很多patch token，模型怎么知道哪个patch原来在图像的什么位置？这就会引出下一章的位置编码问题。
